In [2]:
import kagglehub
import pandas as pd
# Download latest version
path = kagglehub.dataset_download("yeanzc/telco-customer-churn-ibm-dataset")

pd.options.display.max_info_columns=100

pd.set_option('display.max_columns', None)


df = pd.read_excel(f"{path}/Telco_customer_churn.xlsx")
df.describe()


,Count,Zip Code,Latitude,Longitude,Tenure Months,Monthly Charges,Churn Value,Churn Score,CLTV
count,7043.0,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000
mean,1.0,93521.964646,36.282441,-119.798880,32.371149,64.761692,0.265370,58.699418,4400.295755
std,0.0,1865.794555,2.455723,2.157889,24.559481,30.090047,0.441561,21.525131,1183.057152
min,1.0,90001.000000,32.555828,-124.301372,0.000000,18.250000,0.000000,5.000000,2003.000000
25%,1.0,92102.000000,34.030915,-121.815412,9.000000,35.500000,0.000000,40.000000,3469.000000
50%,1.0,93552.000000,36.391777,-119.730885,29.000000,70.350000,0.000000,61.000000,4527.000000
75%,1.0,95351.000000,38.224869,-118.043237,55.000000,89.850000,1.000000,75.000000,5380.500000
max,1.0,96161.000000,41.962127,-114.192901,72.000000,118.750000,1.000000,100.000000,6500.000000


In [3]:
info_custom = pd.DataFrame({
    'Columna': df.columns,
    'DType': df.dtypes,
    'Valores no nulos': df.notna().sum(),
    'Valores nulos': df.isna().sum()
}).reset_index(drop=True)

info_custom

,Columna,DType,Valores no nulos,Valores nulos
0,CustomerID,str,7043,0
1,Count,int64,7043,0
2,Country,str,7043,0
3,State,str,7043,0
4,City,str,7043,0
5,Zip Code,int64,7043,0
6,Lat Long,str,7043,0
7,Latitude,float64,7043,0
8,Longitude,float64,7043,0
9,Gender,str,7043,0


In [4]:
from sqlalchemy import create_engine
import os
import numpy as np
import seaborn as sns
from dotenv import load_dotenv

load_dotenv()

DB_USER=os.getenv("DB_USER")
DB_PASS=os.getenv("DB_PASS")
DB_HOST=os.getenv("DB_HOST", "localhost")
DB_PORT=os.getenv("DB_PORT", "5432")
DB_NAME=os.getenv("DB_NAME", "churnteleco")


df.columns= df.columns.str.lower().str.replace(' ', '_')

url_sql = f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'

engine = create_engine(url_sql)

df.to_sql('customers_raw', engine, schema='raw', if_exists='append', index=False)



113

In [5]:
query_full = 'SELECT * FROM staging.costumers;'

df= pd.read_sql(query_full, engine)

print(df)

     costumer_id  count_costumer        country state_costumer         city  \
0     3668-QPYBK               1  United States     California  Los Angeles   
1     9237-HQITU               1  United States     California  Los Angeles   
2     9305-CDSKC               1  United States     California  Los Angeles   
3     7892-POOKP               1  United States     California  Los Angeles   
4     0280-XJGEX               1  United States     California  Los Angeles   
...          ...             ...            ...            ...          ...   
7038  3213-VVOLG               1  United States     California     Sun City   
7039  2520-SGTTA               1  United States     California   Ben Lomond   
7040  2923-ARZLG               1  United States     California     La Verne   
7041  4075-WKNIU               1  United States     California         Bell   
7042  2775-SEFEE               1  United States     California   Wilmington   

     zip_code                lat_long   latitude   

# Reconocimiento:

Cuantas filas tiene la tabla, y que forma tiene:

In [6]:
query_filas = "SELECT COUNT(*) FROM staging.costumers;"

df = pd.read_sql(query_filas, engine)

print(df)

   count
0   7043


In [7]:
query_base = "SELECT * FROM staging.costumers LIMIT 10;"

df= pd.read_sql(query_base, engine)

print(df)

  costumer_id  count_costumer        country state_costumer         city  \
0  3668-QPYBK               1  United States     California  Los Angeles   
1  9237-HQITU               1  United States     California  Los Angeles   
2  9305-CDSKC               1  United States     California  Los Angeles   
3  7892-POOKP               1  United States     California  Los Angeles   
4  0280-XJGEX               1  United States     California  Los Angeles   
5  4190-MFLUW               1  United States     California  Los Angeles   
6  8779-QRDMV               1  United States     California  Los Angeles   
7  1066-JKSGK               1  United States     California  Los Angeles   
8  6467-CHFZW               1  United States     California  Los Angeles   
9  8665-UTDHZ               1  United States     California  Los Angeles   

  zip_code                lat_long   latitude   longitude  gender  \
0    90003  33.964131, -118.272783  33.964131 -118.272783    Male   
1    90005   34.059281, -

Verificar ids duplicados:

In [8]:
query_duplicados_id = "SELECT costumer_id, COUNT(*) FROM staging.costumers GROUP BY costumer_id HAVING(COUNT(*)) > 1;"

df = pd.read_sql(query_duplicados_id, engine)

print(df)

Empty DataFrame
Columns: [costumer_id, count]
Index: []


Verificar datos duplicados:

In [9]:
query_datos_duplicados = "SELECT *, COUNT(*) FROM staging.costumers GROUP BY costumer_id HAVING(COUNT(*)) > 1"

df = pd.read_sql(query_datos_duplicados, engine)

print(df)

Empty DataFrame
Columns: [costumer_id, count_costumer, country, state_costumer, city, zip_code, lat_long, latitude, longitude, gender, senior_citizen, patner, dependents, tenure_months, phone_service, multiple_lines, internet_service, online_security, online_backup, device_protection, tech_support, streaming_tv, streaming_movies, contract, paperless_billing, payment_method, monthly_charge, total_charges, churn_label, churn_value, churn_score, cltv, churn_reason, count]
Index: []


Buscar datos nulos:

In [10]:
query_datos_null = "SELECT " \
                    "COUNT(*) FILTER (WHERE contract IS NULL) AS null_contract, " \
                    "COUNT(*) FILTER (WHERE total_charges IS NULL) AS null_total_charges, " \
                    "COUNT(*) FILTER (WHERE churn_label IS NULL) AS null_churn_value, " \
                    "COUNT(*) FILTER (WHERE churn_score IS NULL) AS null_churn_score, " \
                    "COUNT(*) FILTER (WHERE cltv IS NULL) AS cltv " \
                    "FROM staging.costumers;"


df = pd.read_sql(query_datos_null, engine)

print(df)

   null_contract  null_total_charges  null_churn_value  null_churn_score  cltv
0              0                   0                 0                 0     0


In [11]:
query_churn_nulos = "SELECT total_charges FROM staging.costumers"  # WHERE total_charges IS NULL;"

df = pd.read_sql(query_churn_nulos, engine)

print(df)

      total_charges
0            108.15
1            151.65
2            820.50
3           3046.05
4           5036.30
...             ...
7038        5976.64
7039        5976.64
7040        5976.64
7041        5976.64
7042        5976.64

[7043 rows x 1 columns]


# Remplazar valores nulos

Utilizamos percentiles al 90%, es decir que el 90% de los datos estan por debajo de dicho percentil, lo cual nos da una perspectiva mas objetiva de los datos
sin la necesidad de usar una media que puede ser sesgada por los valores atipicos.

In [12]:
query_churn_nulos = "SELECT PERCENTILE_CONT(0.9) WITHIN GROUP(ORDER BY total_charges) FROM staging.costumers"  # WHERE total_charges IS NULL;"

df = pd.read_sql(query_churn_nulos, engine)

print(df)


   percentile_cont
0          5976.64


Realizamos un update utilizando sqlalchemy para crear un engine, y poder interactuar con la Base de Datos, utilizamos la consulta anterior para obtener el percentil
90 y lo utilizamos para sustituir los 11 valores nulos encontrados en consultas anteriores.

In [13]:
from sqlalchemy import text


   
query_update = text(
    """ UPDATE staging.costumers 
        SET total_charges= ( SELECT PERCENTILE_CONT(0.9) WITHIN GROUP(ORDER BY total_charges)
        FROM staging.costumers ) 
        WHERE total_charges IS NULL
    """
        )

with engine.begin() as conn:
    conn.execute(query_update)

In [14]:
query_datos_null = "SELECT " \
                    "COUNT(*) FILTER (WHERE contract IS NULL) AS null_contract, " \
                    "COUNT(*) FILTER (WHERE total_charges IS NULL) AS null_total_charges, " \
                    "COUNT(*) FILTER (WHERE churn_label IS NULL) AS null_churn_value, " \
                    "COUNT(*) FILTER (WHERE churn_score IS NULL) AS null_churn_score, " \
                    "COUNT(*) FILTER (WHERE cltv IS NULL) AS cltv " \
                    "FROM staging.costumers;"


df = pd.read_sql(query_datos_null, engine)

print(df)

   null_contract  null_total_charges  null_churn_value  null_churn_score  cltv
0              0                   0                 0                 0     0


Midiendo el balance de los datos, que tantos son SI vs NO.


### Window Function:

Se consulta la cantidad de SI vs NO para analizar el balance de los datos, donde:

1. COUNT(*) -> Total de datos de SI o NO
2. SUM(COUNT) -> Total de datos en el dataset.

La segunda parte no se podria realizar sin una window function dado que esta permite asignar a cada columna el total de datos, permitiendo asi realizar el calculo sobre todo el dataset y con el total de datos de una columna unica.

In [15]:
query_balance_datos = """

    SELECT churn_label, COUNT(*), 
    ROUND( 100 * COUNT(*) / SUM(COUNT(*)) OVER() ,2 ) AS Porcentaje
    FROM staging.costumers
    GROUP BY churn_label;
"""

df=pd.read_sql(query_balance_datos, engine)

print(df)

  churn_label  count  porcentaje
0          No   5174       73.46
1         Yes   1869       26.54


Consultando valores estadisticos:
* Media
* Mediana
* Desviacion Estandar

In [18]:
# NUMERIC COLUMNS:  monthly_charge, total_charges 

query_monthly_change= """
    SELECT 
        MIN(monthly_charge), MAX(monthly_charge),
        ROUND(AVG(monthly_charge)::NUMERIC, 2) AS Media,
        ROUND( (PERCENTILE_CONT(0.5)  WITHIN GROUP (ORDER BY monthly_charge))::NUMERIC,  2) AS Mediana,
        ROUND(STDDEV(monthly_charge)::NUMERIC, 2) AS Desviacion
        
        FROM staging.costumers;
                                
    """



df = pd.read_sql(query_monthly_change, engine)

print (df)


     min     max  media  mediana  desviacion
0  18.25  118.75  64.76    70.35       30.09


In [ ]:
query_total_charges = """
    SELECT 
        MIN(total_charges), MAX(total_charges), 
        ROUND(AVG(total_charges)::NUMERIC, 2) AS media,
        ROUND( (PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY total_charges)) ::NUMERIC, 2) AS mediana,
        ROUND(STDDEV(total_charges)::NUMERIC , 2) AS desviacion_estandar
    FROM staging.costumers;
"""


df= pd.read_sql(query_total_charges, engine)

print(df)

    min     max    media  mediana  desviacion_estandar
0  18.8  8684.8  2289.07  1400.55              2269.69
